# Fitting a Selection Function — CHIME/FRB Selection Function

> Assumes you have the injections JSON (from the Sept 2025 run) and you’ve installed the repo editable so the CLI script can be run.

## 1) Prerequisites

- Python env activated (whatever you used to develop this repo).
- Repo installed editable so relative imports work when running the CLI script:
```bash
pip install -e .
```
- Injections JSON available, e.g.:
```
/data/user-data/kmcgregor/09-2025_injections/output.json
```

## 2) Quick start — 4D baseline fit

This fits the 4D model (`fluence, scattering_time, width, dm`) with a cubic basis (order 3), SNR cut 12, sidelobes excluded beyond \|beam_x\| > 5.0, and saves convergence + histogram plots.

```bash
python ../fitting/logistic_regression_cli.py                                                
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json  
    --base-path ../chimefrb_selection/data/fits  
    --predictor fluence 
    --predictor scattering_time 
    --predictor width 
    --predictor dm   
    --order 3   
    --snr-cut 12   
    --sidelobe-cut 5   
    --plot
```
**Outputs go to:**
```
../chimefrb_selection/data/fits/4d_selection_function/fluence_scattering_time_width_dm/

  IRLS_output_fluence-scattering_time-width-dm_order3_snr12_sl5.0.npz

  plots/irls_convergence_fluence-scattering_time-width-dm_order3_snr12_sl5.0.png

  plots/p_hist_fluence-scattering_time-width-dm_order3_snr12_sl5.0.png

  summary_fluence-scattering_time-width-dm_order3_snr12_sl5.0.csv
```

## 3) 3D / 2D / 1D examples (change the `--predictor` list)

**2D example:** `(fluence, width)` with order 4:
```bash
python ../fitting/logistic_regression_cli.py                             
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json  
    --base-path ../chimefrb_selection/data/fits                          
    --predictor fluence                                                  
    --predictor width                                                    
    --order 4                                                           
    --snr-cut 12                                                         
    --sidelobe-cut 5                                                   
    --plot
```
**3D example:** `(scattering_time, width, dm)` with order 2:
```bash
python ../fitting/logistic_regression_cli.py                                                
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json  
    --base-path ../chimefrb_selection/data/fits                          
    --predictor scattering_time                                          
    --predictor width                                                    
    --predictor dm                                                       
    --order 2                                                            
    --snr-cut 12                                                        
    --sidelobe-cut 5                                                    
    --plot
```

**1D example:** `(scattering_time)` with order 3:
```bash
python ../fitting/logistic_regression_cli.py                             
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json 
    --base-path ../chimefrb_selection/data/fits                          
    --predictor scattering_time                                          
    --order 3                                                            
    --snr-cut 12                                                        
    --sidelobe-cut 5                                                    
    --plot
```
> The **order of predictors** on the command line is preserved in the tag and must match when you later use the model.

## 4) Scan multiple polynomial orders (same predictors & cuts)

Run a sequence of fits for orders 1–5. These commands generate a set of `IRLS_output_*order{N}*.npz` files in the same folder.

```bash
for ORDER in 1 2 3 4 5; do
    python ../fitting/logistic_regression_cli.py                             
        --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json    
        --base-path ../chimefrb_selection/data/fit     
        --predictor fluence 
        --predictor scattering_time 
        --predictor width 
        --predictor dm     
        --order ${ORDER}     
        --snr-cut 12     
        --sidelobe-cut 5     
        --plot
done
```

## 5) Compare saved orders (AIC/BIC, pseudo-R², LRT)

Once you have at least two saved orders for the **same** predictors/SNR/sidelobe settings, run with `--compare-orders` to write comparison tables in CSV.

```bash
python ../fitting/logistic_regression_cli.py                             
        --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json    
        --base-path ../chimefrb_selection/data/fit     
        --predictor fluence 
        --predictor scattering_time 
        --predictor width 
        --predictor dm     
        --order 1     
        --snr-cut 12     
        --sidelobe-cut 5     
        --compare-orders
```

**Writes:**
```
.../model_comparison_fluence-scattering_time-width-dm_snr12_sl5.0.csv
.../likelihood_ratio_test_fluence-scattering_time-width-dm_snr12_sl5.0.csv
```
> The `--order` value on this command is ignored for the comparison itself; it’s used only to build the tag stem and find matching files.

## 6) Rescaled-basis variant (optional)

This additionally fits a min–max **0–1 rescaled** version of the selected **log** predictors and writes a separate CSV of coefficients and SEs. It does *not* overwrite the standard NPZ.

```bash
python ../fitting/logistic_regression_cli.py                             
        --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json    
        --base-path ../chimefrb_selection/data/fit     
        --predictor fluence 
        --predictor scattering_time 
        --predictor width 
        --predictor dm     
        --order 3     
        --snr-cut 12     
        --sidelobe-cut 5     
        --rescaled
        --plot
```
**Writes additionally:**
```
.../IRLS_output_fluence-scattering_time-width-dm_order3_snr12_sl5.0_rescaled.csv
```

## 7) Sidelobe handling

**Exclude sidelobes beyond |beam_x| > 5.0 (default):**
```bash
--sidelobe-cut 5
```

**Use a different threshold, e.g., 3.0:**
```bash
--sidelobe-cut 3
```

**Do not exclude sidelobes:**
```bash
--no-sidelobe-cut
```
> The tag encodes this as `_sl{value}` or `_slnone`. Your choice must be consistent across fits you wish to compare or load later.

## 8) SNR cut variants

The SNR threshold determines which detections are labeled `Y=1` during fitting.

**Examples:**
```bash
--snr-cut 8
--snr-cut 12
--snr-cut 15
```
> The tag includes `_snr{X}`. Changing this creates a different model family.

## 9) Bad-time cut (Sept 11, 2025 gap)

If your pipeline had a known gap (e.g., when L2/L3 missed injections), you can enforce the cut:
```bash
--cut-badtimes      # (this is the default in your script)
```

To disable the cut:
```bash
--no-cut-badtimes
```
> This only affects the **training data** used to fit; it does not change the loader.

## 10) Headless / no plots

If you don’t need convergence or histogram PNGs:
```bash
--no-plot
```

## 11) Caching & refitting

Each fit writes one NPZ named like:
```
IRLS_output_<predictors>_order{N}_snr{X}_sl{cut}.npz
```
If the target NPZ **already exists**, the script loads it instead of refitting.  
To force a refit, **delete** the NPZ and rerun the command.

## 12) Output artifact map

For a tag stem like:
```
fluence-scattering_time-width-dm_order3_snr12_sl5.0
```
you’ll typically get:
```
IRLS_output_<stem>.npz                     # coefficients, std errors, covariance, term names, predictors
summary_<stem>.csv                         # single-row metrics (McFadden R^2, Brier, etc.)
plots/irls_convergence_<stem>.png          # IRLS distance curve
plots/p_hist_<stem>.png                    # p(Detection) histogram for det/nondet
```

## 13) Flag/option reference

| Flag | Purpose | Notes |
|---|---|---|
| `--predictor NAME` (repeatable) | Choose the predictor subset and order | Valid: `fluence`, `scattering_time`, `width`, `dm`. The CLI order is preserved in the tag. |
| `--order N` | Max polynomial degree of the basis | Higher `N` → more terms; watch conditioning. |
| `--snr-cut X` | Label detections with SNR ≥ X as positives | Encoded as `_snrX` in tag. |
| `--sidelobe-cut C` | Exclude sidelobes if \|beam_x\| > C | Encoded as `_slC` in tag. |
| `--no-sidelobe-cut` | Keep sidelobes | Encoded as `_slnone`. |
| `--inj-file PATH` | Path to injections `output.json` | Required. |
| `--base-path DIR` | Root output directory for fits | Script creates subfolders per dimension/predictors. |
| `--rescaled` | Also fit a 0–1 min–max rescaled log-basis | Writes an extra CSV; does **not** replace NPZ. |
| `--compare-orders` | Summarize saved orders (same predictors/cuts) | Writes `model_comparison_*.csv` and `likelihood_ratio_test_*.csv`. |
| `--cut-badtimes / --no-cut-badtimes` | Apply/disable known time gap cut | Train-time only. |
| `--plot / --no-plot` | Enable/disable PNG outputs | Optional. Default plot. |

## 14) Reproducibility recipe

**Run a baseline 4D fit + a small order scan + compare:**


### 1) Baseline
```bash
python ../fitting/logistic_regression_cli.py   \ 
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json   \ 
    --base-path ../chimefrb_selection/data/fits   \ 
    --predictor fluence \ 
    --predictor scattering_time \ 
    --predictor width --predictor dm   \ 
    --order 3 \  
    --snr-cut 12 \  
    --sidelobe-cut 5 \  
    --plot  
```

### 2) Scan orders 1 through 4
```bash
for ORDER in 1 2 3 4; do 
    python ../fitting/logistic_regression_cli.py   \ 
        --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json    \ 
        --base-path ../chimefrb_selection/data/fits   \  
        --predictor fluence \ 
        --predictor scattering_time \ 
        --predictor width \ 
        --predictor dm     \ 
        --order ${ORDER} \ 
        --snr-cut 12 \ 
        --sidelobe-cut 5 \ 
        --plot 
done 
```

### 3) Summarize
```bash 
python ../fitting/logistic_regression_cli.py \ 
    --inj-file /data/user-data/kmcgregor/09-2025_injections/output.json  \ 
    --base-path ../chimefrb_selection/data/fits    \ 
    --predictor fluence \ 
    --predictor scattering_time \ 
    --predictor width \ 
    --predictor dm   \ 
    --order 1 \ 
    --snr-cut 12 \ 
    --sidelobe-cut 5 \ 
    --compare-orders  
```